In [62]:
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpointEmbeddings
import os

In [54]:
loader = YoutubeLoader(video_id='X8ZD7nTQ5iI')

In [55]:
doc = loader.load()

In [66]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0
)

In [67]:
chunks = splitter.create_documents([doc[0].page_content])

In [68]:
load_dotenv()

HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_ACCESS_TOKEN')

embedding = HuggingFaceEndpointEmbeddings(
    model='google/embeddinggemma-300m',
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN
)

In [69]:
vectorstore = FAISS.from_documents(chunks, embedding)

In [91]:
retriever = vectorstore.as_retriever(search_kwargs={'k':10})

In [92]:
prmpt = PromptTemplate(
    template="""
you are helpful assistant, answer ONLY from provided transcript context.
if context is insufficient, then just say I don't know. Context -

{context}

Question : {question}
""",
input_variables=['context', 'question']
)

In [100]:
question = 'what is recent good development in renewable energy?'

retrieved_docs = retriever.invoke(question)

In [101]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [102]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [103]:
final_prompt = prmpt.invoke({'context': context_text, 'question': question})

In [105]:
model.invoke(final_prompt)

AIMessage(content='Renewable energy has surpassed coal globally for the first time, with renewables supplying over 5,000 terawatt hours worldwide between January and June.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 463, 'total_tokens': 495, 'completion_time': 0.043898736, 'completion_tokens_details': None, 'prompt_time': 0.025683661, 'prompt_tokens_details': None, 'queue_time': 0.055373289, 'total_time': 0.069582397}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b638a-8591-76c1-9f3d-da222e54af7c-0', usage_metadata={'input_tokens': 463, 'output_tokens': 32, 'total_tokens': 495})

In [106]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [107]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [108]:
parallel_chain = RunnableParallel(
   {
        'context' : retriever | RunnableLambda(format_docs),
        'question' : RunnablePassthrough()
   }
)

In [113]:
parser = StrOutputParser()

In [114]:
final_chain = parallel_chain | prmpt | model | parser

In [118]:
print(final_chain.invoke('Major Wins for Forest Protection Around the World'))

2025 saw major wins for forest protection around the world.
